# Math 189 Final Project
## Can Neighborhood Wealth Predict County-Level Standardized Test Scores in California?

**Data Sources:** CAASPP (2024-25), CDE FRPM (2024-25), IPUMS ACS 5-Year (2023)

---

### Table of Contents
1. Imports & Setup
2. Data Acquisition — CAASPP Test Scores
3. Data Acquisition — FRPM Poverty Data
4. Data Acquisition — IPUMS ACS Census Data
5. Data Merging & Cleaning
6. Exploratory Data Analysis (EDA)
7. California County Heatmap
8. Two-Sample KS Test
9. VIF Analysis
10. Variable Selection
11. Variance Decomposition
12. Linear Regression
13. Regression Diagnostics

In [2]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import gzip
import os
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Patch

# Plot settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'


In [3]:
# Load raw CAASPP file from zip archive
with zipfile.ZipFile('sb_ca2025_1_csv_v1.zip', 'r') as z:
    with z.open('sb_ca2025_1_csv_v1.txt') as f:
        df_raw = pd.read_csv(f, delimiter='^', dtype=str, low_memory=False, encoding='latin-1')

print(f"Raw CAASPP data: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")

Raw CAASPP data: 102,044 rows, 69 columns


In [4]:
# Filter to county-level ELA and Math
county = df_raw[
    (df_raw['Type ID'] == '5') &
    (df_raw['Grade'] == '13')
].copy()

county = county[['County Code', 'Test ID', 'Total Students Tested', 'Percentage Standard Met and Above']]
county['Percentage Standard Met and Above'] = pd.to_numeric(county['Percentage Standard Met and Above'], errors='coerce')
county['Total Students Tested'] = pd.to_numeric(county['Total Students Tested'], errors='coerce')

county_wide = county.pivot_table(
    index='County Code',
    columns='Test ID',
    values='Percentage Standard Met and Above',
    aggfunc='first'
).reset_index()

county_wide.columns.name = None
county_wide = county_wide.rename(columns={'1': 'pct_met_ELA', '2': 'pct_met_Math'})

In [5]:
# Add county names
with zipfile.ZipFile('sb_ca2025_1_csv_v1.zip', 'r') as z:
    with z.open('sb_ca2025entities_csv.txt') as f:
        entities = pd.read_csv(f, delimiter='^', dtype=str, low_memory=False, encoding='latin-1')

county_names = entities[entities['Type ID'] == '5'][['County Code', 'County Name']].drop_duplicates()
county_wide = county_wide.merge(county_names, on='County Code', how='left')
county_wide = county_wide[['County Code', 'County Name', 'pct_met_ELA', 'pct_met_Math']]

In [6]:
print(county_wide.to_string(index=False))

County Code     County Name  pct_met_ELA  pct_met_Math
         01         Alameda        56.16         48.15
         02          Alpine        43.24         29.41
         03          Amador        35.79         23.49
         04           Butte        46.38         33.23
         05       Calaveras        34.21         23.18
         06          Colusa        35.82         23.29
         07    Contra Costa        51.98         42.02
         08       Del Norte        34.02         21.13
         09       El Dorado        57.75         46.07
         10          Fresno        46.44         34.09
         11           Glenn        31.92         21.68
         12        Humboldt        42.54         31.92
         13        Imperial        41.59         27.11
         14            Inyo        35.75         28.70
         15            Kern        39.94         24.26
         16           Kings        46.38         31.99
         17            Lake        26.07         15.87
         1